# Neural Network Model

В данном ноутбуке выполняется обучение нейронной сети для задачи предсказания вероятности взаимодействия пользователя с постом.

Данные используются после этапа Feature Engineering.

На данном этапе:
- загружаются X_train, y_train, X_test, y_test;
- создаётся архитектура нейронной сети;
- выполняется обучение модели;
- оценивается качество по ROC-AUC;

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from sklearn.preprocessing import StandardScaler

X_train = pd.read_csv("../data/processed/X_train.csv")
X_test = pd.read_csv("../data/processed/X_test.csv")
y_test = pd.read_csv("../data/processed/y_test.csv")
y_train = pd.read_csv("../data/processed/y_train.csv")

## Подготовка данных для нейронной сети

Перед обучением нейронной сети выполняется масштабирование признаков.

Так как нейронные сети чувствительны к масштабу входных данных, применяется `StandardScaler`:
- параметры масштабирования вычисляются только на обучающей выборке;
- тестовая выборка преобразуется с использованием тех же параметров, чтобы избежать утечки данных.

In [ ]:
scaler = StandardScaler()
X_train_nn = scaler.fit_transform(X_train)
X_test_nn = scaler.transform(X_test)

## Преобразование данных в Tensor

После масштабирования признаки преобразуются в формат `Tensor`, необходимый для обучения нейронной сети в PyTorch.

Входные признаки и целевая переменная переводятся в формат `float32`, так как данный тип используется для эффективного обучения нейронных сетей.

In [ ]:
X_train_nn = torch.tensor(X_train_nn)
y_train_nn = torch.tensor(y_train.values.astype(np.float32))

X_test_nn = torch.tensor(X_test_nn)
y_test_nn = torch.tensor(y_test.values.astype(np.float32))

## Архитектура нейронной сети RecMLP

Для задачи предсказания вероятности взаимодействия пользователя с постом используется полносвязная нейронная сеть (MLP).

Архитектура модели включает:
- несколько линейных слоёв для изучения зависимостей между признаками;
- Batch Normalization для стабилизации обучения;
- ReLU как функцию активации;
- Dropout для уменьшения переобучения;
- выходной слой с одним нейроном для задачи бинарной классификации.

На выходе модель возвращает логиты, которые далее используются с функцией потерь `BCEWithLogitsLoss`.

In [ ]:
class RecMLP(nn.Module):
  def __init__(self, input_dim):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(128, 1)
        )
  def forward(self, x):
        return self.net(x).squeeze()

## Приведение Tensor к нужному формату

Перед обучением модели выполняется финальная подготовка Tensor:

- признаки переводятся в формат `float32`, который используется PyTorch для обучения нейронных сетей;
- целевые значения приводятся к одномерному виду, необходимому для функции потерь `BCEWithLogitsLoss`.

Это позволяет избежать ошибок несовпадения типов и размерностей при обучении.

In [ ]:
X_train_nn = X_train_nn.float()
X_test_nn = X_test_nn.float()
y_train_nn = y_train_nn.view(-1)
y_test_nn = y_test_nn.view(-1)

## Создание Dataset и DataLoader

Для обучения нейронной сети данные преобразуются в формат PyTorch Dataset.

`TensorDataset` объединяет признаки и целевую переменную в пары `(features, target)`.

`DataLoader` используется для:
- обучения модели батчами вместо загрузки всей выборки в память;
- перемешивания данных во время обучения (`shuffle=True`);
- более эффективной работы с большими объёмами данных.

In [ ]:
from torch.utils.data import TensorDataset, DataLoader

train_ds = TensorDataset(X_train_nn, y_train_nn)
train_loader = DataLoader(train_ds, batch_size=1024, shuffle=True)

## Обучение нейронной сети

На данном этапе выполняется обучение модели RecMLP.

Используется:
- `BCEWithLogitsLoss` в качестве функции потерь для бинарной классификации;
- оптимизатор `Adam` для обновления весов модели;
- `weight_decay` для дополнительной регуляризации и снижения риска переобучения;
- GPU (`CUDA`), если он доступен, для ускорения обучения.

Обучение выполняется в течение нескольких эпох с использованием батчей из `DataLoader`.

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = RecMLP(X_train_nn.shape[1]).to(device)

criterion = nn.BCEWithLogitsLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=5e-4,
    weight_decay=1e-5
)
for epoch in range(10):
    model.train()
    total_loss = 0

    for X_batch, y_batch in train_loader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()

        preds = model(X_batch)

        loss = criterion(preds, y_batch)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"epoch {epoch}: loss={total_loss / len(train_loader):.4f}")

epoch 0: loss=0.3053
epoch 1: loss=0.3012
epoch 2: loss=0.2999
epoch 3: loss=0.2992
epoch 4: loss=0.2987
epoch 5: loss=0.2984
epoch 6: loss=0.2981
epoch 7: loss=0.2979
epoch 8: loss=0.2977
epoch 9: loss=0.2976


## Оценка качества нейронной сети

После завершения обучения выполняется оценка модели на обучающей и тестовой выборках.

Для получения предсказаний:
- модель переводится в режим `evaluation` через `model.eval()`;
- отключается расчёт градиентов с помощью `torch.no_grad()`;
- данные обрабатываются батчами для снижения потребления памяти.

Качество модели оценивается по метрике ROC-AUC, что позволяет сравнить нейронную сеть с предыдущими моделями (Logistic Regression, CatBoost, LightGBM).

In [ ]:
def get_preds(model, X):
    model.eval()
    preds = []

    with torch.no_grad():
        for i in range(0, len(X), 1024):
            batch = X[i:i+1024].to(device)
            out = model(batch)
            preds.append(out.cpu())

    return torch.cat(preds).numpy()

from sklearn.metrics import roc_auc_score

train_preds = get_preds(model, X_train_nn)

train_auc = roc_auc_score(y_train_nn.cpu().numpy(), train_preds)

print("Train ROC-AUC:", train_auc)

test_preds = get_preds(model, X_test_nn)

test_auc = roc_auc_score(y_test_nn.cpu().numpy(), test_preds)

print("Test ROC-AUC:", test_auc)

Train ROC-AUC: 0.7160076026757926
Test ROC-AUC: 0.6576304528539368


## Сохранение модели

После обучения веса нейронной сети сохраняются в файл `.pth`.

Сохраняется только `state_dict` модели, который содержит обученные параметры нейронной сети. Архитектура модели будет восстановлена при загрузке через класс `RecMLP`.

In [ ]:
torch.save(model.state_dict(), "../models/neural_network.pth")

## Сохранение scaler

Так как перед подачей данных в нейронную сеть используется `StandardScaler`, его параметры также сохраняются.

Это необходимо для дальнейшего инференса: новые данные должны быть преобразованы точно так же, как и обучающая выборка.

In [ ]:
import joblib

joblib.dump(
    scaler,
    "../models/scaler.pkl"
)